### Plot Generation Workflow

Purpose: Collect the publication-facing plotting cells in one notebook so the manuscript figures can be regenerated from the cleaned data and saved artifacts.

Notebook map:
- `test.ipynb`: full tabular workflow on the curated dataset
- `test_cluster1.ipynb`: regression-mixture, Cluster 1 cleaning, and transfer workflow
- `test_gcn.ipynb`: shielding-aware GCN workflow
- `draw.ipynb`: centralized manuscript plotting workflow

Path convention:
- `data/raw`: original input datasets
- `data/processed`: cleaned datasets and engineered features
- `results`: figures, tables, models, and GCN artifacts


### Experimental Shift Distribution

Purpose: Recreate the curated experimental 19F chemical-shift histogram used as the dataset overview.


In [ ]:
# -----------------------------------------------------------------------------
# Plot the experimental 19F chemical-shift distribution
# Purpose: Load the curated shift dataset and export the histogram used as the dataset overview.
# -----------------------------------------------------------------------------

from pathlib import Path
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

from plotting_standard import (
    apply_plot_style,
    create_single_axis_figure,
    prepare_display_table,
    print_key_value_rows,
    print_section,
    report_saved_paths,
    save_figure,
)

PROJECT_ROOT = Path(".").resolve()
SHIFT_CSV_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_shift_data.csv"
FIG_DIR = PROJECT_ROOT / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

BLUE = "#4C78A8"
AXIS_LABEL_SIZE = 17.5
TICK_LABEL_SIZE = 16.5
NOTE_SIZE = 16.5

apply_plot_style(profile="paper")

if not SHIFT_CSV_PATH.exists():
    raise FileNotFoundError(
        f"Shift CSV file not found:\n  {SHIFT_CSV_PATH}\n"
        "Please place the file at data/processed/cleaned_shift_data.csv before running this cell."
    )

df = pd.read_csv(SHIFT_CSV_PATH)
required_cols = {"SMILES", "shift_value"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(
        f"Missing required columns: {missing}. Found columns: {list(df.columns)}"
    )

df["SMILES"] = df["SMILES"].astype(str).str.strip()
df["shift_value"] = pd.to_numeric(df["shift_value"], errors="coerce")

n_before = len(df)
df = df.dropna(subset=["SMILES", "shift_value"]).reset_index(drop=True)
n_after = len(df)

print_section("Input dataset overview")
print_key_value_rows(
    [
        ("Project root", PROJECT_ROOT),
        ("Input file", SHIFT_CSV_PATH),
        ("Rows before cleaning", n_before),
        ("Rows after cleaning", n_after),
    ]
)
try:
    display(prepare_display_table(df.head()))
except NameError:
    print(prepare_display_table(df.head()).to_string(index=False))

values = df["shift_value"].to_numpy()
n = len(values)

q25, q75 = np.percentile(values, [25, 75])
iqr = q75 - q25
if iqr > 0:
    bin_width = 2 * iqr * (n ** (-1 / 3))
    bins = math.ceil((values.max() - values.min()) / bin_width) if bin_width > 0 else 30
else:
    bins = int(np.sqrt(n))
bins = max(22, min(bins, 38))

print_section("Histogram settings")
print_key_value_rows(
    [
        ("Final sample size (n)", n),
        ("Number of bins", bins),
        ("Minimum shift / ppm", f"{values.min():.2f}"),
        ("Maximum shift / ppm", f"{values.max():.2f}"),
    ]
)

fig, ax = create_single_axis_figure("single_wide", profile="paper")
counts, _, _ = ax.hist(
    values,
    bins=bins,
    color=BLUE,
    edgecolor="black",
    linewidth=0.55,
)

ax.set_xlabel(r"$\delta_{\mathrm{exp}}$ (ppm)", labelpad=2, fontsize=AXIS_LABEL_SIZE)
ax.set_ylabel("Count", labelpad=2, fontsize=AXIS_LABEL_SIZE)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.7)

ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.tick_params(axis="both", which="major", pad=1.5, labelsize=TICK_LABEL_SIZE)
ax.set_ylim(0, max(counts) * 1.08)

ax.text(
    0.97,
    0.96,
    f"n = {n}",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=NOTE_SIZE,
)

fig.tight_layout(pad=0.5)
saved_paths = save_figure(fig, FIG_DIR / "Figure_1_shift_distribution.png")
plt.show()
report_saved_paths(saved_paths, "Saved figure files")


### Molecular-Weight Distribution

Purpose: Plot the molecular-weight distribution of the curated fluorinated molecules.


In [ ]:
# -----------------------------------------------------------------------------
# Plot the molecular-weight distribution of the curated dataset
# Purpose: Calculate molecular weights from SMILES and export the molecular-weight distribution plot.
# -----------------------------------------------------------------------------

from pathlib import Path
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors
except ImportError as exc:
    raise ImportError(
        "This cell requires RDKit in the notebook environment."
    ) from exc

from plotting_standard import (
    create_single_axis_figure,
    print_key_value_rows,
    print_section,
    report_saved_paths,
    save_figure,
)

PROJECT_ROOT = Path(".").resolve()
CURATED_SHIFT_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_shift_data.csv"
FIG_DIR = PROJECT_ROOT / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

BLUE = "#4C78A8"
AXIS_LABEL_SIZE = 17.5
TICK_LABEL_SIZE = 16.5
NOTE_SIZE = 16.5


def compute_fd_bins(values, min_bins=18, max_bins=36):
    values = np.asarray(values, dtype=float)
    q25, q75 = np.percentile(values, [25, 75])
    iqr = q75 - q25
    n = len(values)

    if iqr > 0:
        bin_width = 2 * iqr * (n ** (-1 / 3))
        bins = math.ceil((values.max() - values.min()) / bin_width) if bin_width > 0 else min_bins
    else:
        bins = int(np.sqrt(n))

    return max(min_bins, min(bins, max_bins))


if not CURATED_SHIFT_PATH.exists():
    raise FileNotFoundError(
        f"Curated shift CSV file not found:\n  {CURATED_SHIFT_PATH}\n"
        "Please place the file at data/processed/cleaned_shift_data.csv before running this cell."
    )

curated_df = pd.read_csv(CURATED_SHIFT_PATH)
if "SMILES" not in curated_df.columns:
    raise ValueError(
        f"Missing required column 'SMILES'. Found columns: {list(curated_df.columns)}"
    )

curated_df["SMILES"] = curated_df["SMILES"].astype(str).str.strip()
curated_df = curated_df.dropna(subset=["SMILES"]).drop_duplicates(subset=["SMILES"]).reset_index(drop=True)

molwt_records = []
invalid_smiles = []

for smiles in curated_df["SMILES"]:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        invalid_smiles.append(smiles)
        continue
    molwt_records.append({"SMILES": smiles, "MolWt": Descriptors.MolWt(mol)})

if invalid_smiles:
    raise ValueError(
        "RDKit failed to parse some curated SMILES entries. "
        f"Examples: {invalid_smiles[:5]}"
    )

molwt_df = pd.DataFrame(molwt_records)
molwt_values = molwt_df["MolWt"].to_numpy()
molwt_bins = compute_fd_bins(molwt_values)

print_section("Molecular-weight overview")
print_key_value_rows(
    [
        ("Curated molecules (n)", len(molwt_df)),
        ("Number of bins", molwt_bins),
        (
            "Molecular weight range / g mol^-1",
            f"{molwt_df['MolWt'].min():.2f} to {molwt_df['MolWt'].max():.2f}",
        ),
    ]
)

fig, ax = create_single_axis_figure("single_wide", profile="paper")
counts, _, _ = ax.hist(
    molwt_values,
    bins=molwt_bins,
    color=BLUE,
    edgecolor="black",
    linewidth=0.55,
)

ax.set_xlabel(r"Molecular weight (g mol$^{-1}$)", labelpad=2, fontsize=AXIS_LABEL_SIZE)
ax.set_ylabel("Count", labelpad=2, fontsize=AXIS_LABEL_SIZE)
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.7)

ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.tick_params(axis="both", which="major", pad=1.5, labelsize=TICK_LABEL_SIZE)
ax.set_ylim(0, max(counts) * 1.08)
ax.text(
    0.97,
    0.96,
    f"n = {len(molwt_df)}",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=NOTE_SIZE,
)

fig.tight_layout(pad=0.5)
saved_paths = save_figure(fig, FIG_DIR / "Figure_1_molecular_weight_distribution.png")
plt.show()
report_saved_paths(saved_paths, "Saved figure files")


### Fluorine-Environment Distribution

Purpose: Summarize the fluorine environments represented in the curated dataset.


In [ ]:
# -----------------------------------------------------------------------------
# Summarize fluorine environments in the curated dataset
# Purpose: Classify fluorine environments from the molecular graph and export the distribution plot.
# -----------------------------------------------------------------------------

from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from rdkit import Chem
except ImportError as exc:
    raise ImportError(
        "This cell requires RDKit in the notebook environment."
    ) from exc

from plotting_standard import (
    apply_plot_style,
    print_key_value_rows,
    print_section,
    report_saved_paths,
    save_figure,
)

PROJECT_ROOT = Path(".").resolve()
CURATED_SHIFT_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_shift_data.csv"
FIG_DIR = PROJECT_ROOT / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

COLOR_MAP = {
    "Aliphatic": "#4C78A8",
    "Ring-bound": "#F28E2B",
    "Aromatic": "#59A14F",
    "Heteroatom-bound": "#E15759",
}

MAIN_PERCENT_SIZE = 17.5
INSET_TITLE_SIZE = 17.0
INSET_TEXT_SIZE = 16.5
LEGEND_TEXT_SIZE = 17.0
LEGEND_TITLE_SIZE = 17.0
BOTTOM_TEXT_SIZE = 17.0


def classify_f_environment(f_atom):
    neighbors = f_atom.GetNeighbors()
    if len(neighbors) != 1:
        return "Other"

    anchor = neighbors[0]
    if anchor.GetIsAromatic():
        return "Aromatic"
    if anchor.IsInRing():
        return "Ring-bound"
    if anchor.GetAtomicNum() == 6:
        return "Aliphatic"
    return "Heteroatom-bound"


def compute_display_percentages(counts: pd.Series) -> dict[str, float]:
    total = int(counts.sum())
    labels = list(counts.index)
    if not labels:
        return {}

    if len(labels) == 1:
        return {labels[0]: 100.00}

    display = {}
    running = 0.0
    for label in labels[:-1]:
        value = round(counts[label] / total * 100.0, 2)
        display[label] = value
        running += value

    display[labels[-1]] = round(100.0 - running, 2)
    return display


if not CURATED_SHIFT_PATH.exists():
    raise FileNotFoundError(
        f"Curated shift CSV file not found:\n  {CURATED_SHIFT_PATH}\n"
        "Please place the file at data/processed/cleaned_shift_data.csv before running this cell."
    )

curated_df = pd.read_csv(CURATED_SHIFT_PATH)
if "SMILES" not in curated_df.columns:
    raise ValueError(
        f"Missing required column 'SMILES'. Found columns: {list(curated_df.columns)}"
    )

curated_df["SMILES"] = curated_df["SMILES"].astype(str).str.strip()
curated_df = curated_df.dropna(subset=["SMILES"]).drop_duplicates(subset=["SMILES"]).reset_index(drop=True)

environment_counter = Counter()
invalid_smiles = []

for smiles in curated_df["SMILES"]:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        invalid_smiles.append(smiles)
        continue

    for atom in mol.GetAtoms():
        if atom.GetAtomicNum() == 9:
            environment_counter[classify_f_environment(atom)] += 1

if invalid_smiles:
    raise ValueError(
        "RDKit failed to parse some curated SMILES entries. "
        f"Examples: {invalid_smiles[:5]}"
    )

environment_order = ["Aliphatic", "Ring-bound", "Aromatic", "Heteroatom-bound"]
environment_counts = pd.Series(
    {label: environment_counter.get(label, 0) for label in environment_order},
    dtype=int,
)
environment_counts = environment_counts[environment_counts > 0]
total_f_atoms = int(environment_counts.sum())
display_percentages = compute_display_percentages(environment_counts)

print_section("Fluorine-environment overview")
print_key_value_rows(
    [
        ("Curated molecules (n)", len(curated_df)),
        ("Total F atoms counted", total_f_atoms),
    ]
)
for label, count in environment_counts.items():
    print(f"- {label:17s}: {count} ({display_percentages[label]:.2f}%)")

apply_plot_style(profile="paper")
fig = plt.figure(figsize=(7.8, 4.8), dpi=150)
ax = fig.add_axes([0.08, 0.08, 0.57, 0.84])

wedges, _ = ax.pie(
    environment_counts.to_numpy(),
    labels=None,
    colors=[COLOR_MAP[label] for label in environment_counts.index],
    startangle=90,
    counterclock=False,
    wedgeprops={"linewidth": 0.7, "edgecolor": "white"},
)

for idx, label in enumerate(environment_counts.index):
    if display_percentages[label] < 4.0:
        continue
    wedge = wedges[idx]
    theta = np.deg2rad((wedge.theta1 + wedge.theta2) / 2.0)
    x = np.cos(theta) * 0.62
    y = np.sin(theta) * 0.62
    ax.text(
        x,
        y,
        f"{display_percentages[label]:.2f}%",
        color="white",
        fontsize=MAIN_PERCENT_SIZE,
        fontweight="bold",
        ha="center",
        va="center",
    )

minor_labels = [label for label in environment_counts.index if display_percentages[label] < 4.0]
if minor_labels:
    minor_counts = environment_counts[minor_labels]
    inset_ax = fig.add_axes([0.61, 0.60, 0.27, 0.27])
    inset_ax.pie(
        minor_counts.to_numpy(),
        colors=[COLOR_MAP[label] for label in minor_counts.index],
        startangle=90,
        counterclock=False,
        explode=[0.18 if label == "Heteroatom-bound" else 0.0 for label in minor_counts.index],
        wedgeprops={"linewidth": 0.5, "edgecolor": "white"},
    )
    inset_ax.set_aspect("equal")
    inset_ax.set_title("Minor classes\n(magnified)", fontsize=INSET_TITLE_SIZE, pad=2)
    inset_text = "\n".join(
        f"{label.replace('-bound', '')} {display_percentages[label]:.2f}%"
        for label in minor_counts.index
    )
    inset_ax.text(
        0.0,
        -1.18,
        inset_text,
        ha="center",
        va="top",
        fontsize=INSET_TEXT_SIZE,
    )

legend_labels = [
    f"{label}: {count} ({display_percentages[label]:.2f}%)"
    for label, count in environment_counts.items()
]

ax.legend(
    wedges,
    legend_labels,
    title="F environment",
    loc="center left",
    bbox_to_anchor=(0.95, 0.24),
    frameon=False,
    fontsize=LEGEND_TEXT_SIZE,
    title_fontsize=LEGEND_TITLE_SIZE,
)
ax.set_aspect("equal")
ax.text(
    0.0,
    -1.18,
    f"Total F atoms = {total_f_atoms}",
    ha="center",
    va="top",
    fontsize=BOTTOM_TEXT_SIZE,
)

saved_paths = save_figure(fig, FIG_DIR / "Figure_1_fluorine_environment_distribution_pie.png")
plt.show()
report_saved_paths(saved_paths, "Saved figure files")


### Fluorine Count Distribution

Purpose: Show how many fluorine atoms appear in each molecule of the curated dataset.


In [ ]:
# -----------------------------------------------------------------------------
# Plot the fluorine count distribution per molecule
# Purpose: Count fluorine atoms per molecule and export the fluorine-count distribution plot.
# -----------------------------------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

try:
    from rdkit import Chem
except ImportError as exc:
    raise ImportError(
        "This cell requires RDKit in the notebook environment."
    ) from exc

from plotting_standard import (
    create_single_axis_figure,
    print_key_value_rows,
    print_section,
    report_saved_paths,
    save_figure,
)

PROJECT_ROOT = Path(".").resolve()
CURATED_SHIFT_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_shift_data.csv"
FIG_DIR = PROJECT_ROOT / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

ORANGE = "#F28E2B"
AXIS_LABEL_SIZE = 17.5
TICK_LABEL_SIZE = 16.5
NOTE_SIZE = 16.5

if not CURATED_SHIFT_PATH.exists():
    raise FileNotFoundError(
        f"Curated shift CSV file not found:\n  {CURATED_SHIFT_PATH}\n"
        "Please place the file at data/processed/cleaned_shift_data.csv before running this cell."
    )

curated_df = pd.read_csv(CURATED_SHIFT_PATH)
if "SMILES" not in curated_df.columns:
    raise ValueError(
        f"Missing required column 'SMILES'. Found columns: {list(curated_df.columns)}"
    )

curated_df["SMILES"] = curated_df["SMILES"].astype(str).str.strip()
curated_df = curated_df.dropna(subset=["SMILES"]).drop_duplicates(subset=["SMILES"]).reset_index(drop=True)

f_counts = []
invalid_smiles = []

for smiles in curated_df["SMILES"]:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        invalid_smiles.append(smiles)
        continue
    f_counts.append(sum(atom.GetAtomicNum() == 9 for atom in mol.GetAtoms()))

if invalid_smiles:
    raise ValueError(
        "RDKit failed to parse some curated SMILES entries. "
        f"Examples: {invalid_smiles[:5]}"
    )

f_count_distribution = pd.Series(f_counts, name="F_count").value_counts().sort_index()

print_section("Fluorine-count overview")
print_key_value_rows(
    [
        ("Curated molecules (n)", len(f_counts)),
        ("Maximum F atoms in one molecule", int(max(f_counts))),
    ]
)

fig, ax = create_single_axis_figure("single_wide", profile="paper")
x = np.arange(len(f_count_distribution))
bars = ax.bar(
    x,
    f_count_distribution.to_numpy(),
    color=ORANGE,
    edgecolor="black",
    linewidth=0.55,
)

ax.set_xlabel("Number of F atoms per molecule", labelpad=2, fontsize=AXIS_LABEL_SIZE)
ax.set_ylabel("Count", labelpad=2, fontsize=AXIS_LABEL_SIZE)
ax.set_xticks(x, [str(value) for value in f_count_distribution.index])
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.tick_params(axis="both", which="major", pad=1.5, labelsize=TICK_LABEL_SIZE)
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.7)

ymax = max(f_count_distribution.to_numpy())
ax.set_ylim(0, ymax * 1.12)
for bar, value in zip(bars, f_count_distribution.to_numpy()):
    ax.text(
        bar.get_x() + bar.get_width() / 2.0,
        value + ymax * 0.015,
        f"{int(value)}",
        ha="center",
        va="bottom",
        fontsize=NOTE_SIZE,
    )

ax.text(
    0.97,
    0.96,
    f"n = {len(f_counts)}",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=NOTE_SIZE,
)
fig.tight_layout(pad=0.5)

saved_paths = save_figure(fig, FIG_DIR / "Figure_1_fluorine_count_distribution.png")
plt.show()
report_saved_paths(saved_paths, "Saved figure files")


### Classification Confusion Matrix

Purpose: Visualize the test-set confusion matrix for the structural classifier that separates the two regimes.


In [ ]:
# -----------------------------------------------------------------------------
# Plot the classification confusion matrix
# Purpose: Train the structural classifier on the saved feature sets and export the test-set confusion matrix.
# -----------------------------------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

from plotting_standard import (
    apply_plot_style,
    print_key_value_rows,
    print_section,
    report_saved_paths,
    save_figure,
)

PROJECT_ROOT = Path(".").resolve()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
FIG_DIR = PROJECT_ROOT / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42

ALL_FEATURES_PATH = DATA_DIR / "features_dataset_with_shield.csv"
CLASS1_FEATURES_PATH = DATA_DIR / "features_dataset_with_shield_cluster1.csv"

AXIS_LABEL_SIZE = 15.5
TICK_LABEL_SIZE = 14.5
NOTE_SIZE = 14.5


def build_classification_dataset():
    if not ALL_FEATURES_PATH.exists():
        raise FileNotFoundError(f"Full feature dataset not found: {ALL_FEATURES_PATH}")
    if not CLASS1_FEATURES_PATH.exists():
        raise FileNotFoundError(f"Cluster 1 feature dataset not found: {CLASS1_FEATURES_PATH}")

    df_all = pd.read_csv(ALL_FEATURES_PATH)
    df_class1 = pd.read_csv(CLASS1_FEATURES_PATH)

    candidate_keys = ["SMILES", "smiles", "ID", "id", "Name", "name"]
    match_key = None
    for key in candidate_keys:
        if key in df_all.columns and key in df_class1.columns:
            match_key = key
            break
    if match_key is None:
        raise ValueError("No shared matching key found between the full dataset and the cluster1 dataset.")

    df_all[match_key] = df_all[match_key].astype(str).str.strip()
    df_class1[match_key] = df_class1[match_key].astype(str).str.strip()
    class1_keys = set(df_class1[match_key].dropna().astype(str))
    df_all["target_class"] = df_all[match_key].astype(str).isin(class1_keys).astype(int)

    exclude_cols = {
        "target_class", "shift_value", match_key,
        "SMILES", "smiles", "Name", "name", "ID", "id",
        "cluster", "Cluster", "class", "Class", "priority_class", "reg_cluster",
    }
    feature_cols = [
        col for col in df_all.columns
        if col not in exclude_cols and pd.api.types.is_numeric_dtype(df_all[col])
    ]

    X = df_all[feature_cols].replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True)).fillna(0)
    y = df_all["target_class"].astype(int)
    return X, y


def train_classifier():
    X, y = build_classification_dataset()
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=SEED,
        stratify=y,
    )
    clf = ExtraTreesClassifier(
        n_estimators=500,
        max_features="sqrt",
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1,
    )
    clf.fit(X_train, y_train)
    return clf, X_test, y_test


apply_plot_style(profile="paper")
clf, X_test, y_test = train_classifier()
y_pred = clf.predict(X_test)
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
accuracy = float(accuracy_score(y_test, y_pred))

print_section("Classification confusion-matrix summary")
print_key_value_rows([
    ("Test samples", len(y_test)),
    ("Misclassified samples", int((y_test != y_pred).sum())),
    ("Accuracy", f"{accuracy:.3f}"),
])

fig, ax = plt.subplots(figsize=(5.2, 5.0), dpi=150)
ax.imshow(cm.astype(float), cmap="Blues", vmin=0, vmax=max(cm.max(), 1))

for row in range(cm.shape[0]):
    for col in range(cm.shape[1]):
        value = cm[row, col]
        color = "white" if value >= 0.55 * cm.max() else "black"
        ax.text(
            col,
            row,
            f"{value}",
            ha="center",
            va="center",
            fontsize=NOTE_SIZE,
            fontweight="bold",
            color=color,
        )

tick_labels = ["Class 0\n(off-band)", "Class 1\n(main-band)"]
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(tick_labels)
ax.set_yticklabels(tick_labels)
ax.set_xlabel("Predicted label", fontsize=AXIS_LABEL_SIZE, labelpad=8)
ax.set_ylabel("True label", fontsize=AXIS_LABEL_SIZE, labelpad=8)
ax.tick_params(axis="both", which="major", labelsize=TICK_LABEL_SIZE, pad=4)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.8)

ax.text(
    0.98,
    0.02,
    f"Accuracy = {accuracy:.3f}",
    transform=ax.transAxes,
    ha="right",
    va="bottom",
    fontsize=NOTE_SIZE,
)

fig.tight_layout(pad=0.65)
saved_paths = save_figure(fig, FIG_DIR / "Figure_4a_classification_confusion_matrix.png")
plt.show()
report_saved_paths(saved_paths, "Saved figure files")


### Classification ROC Curve

Purpose: Visualize the ROC performance of the structural classifier.


In [ ]:
# -----------------------------------------------------------------------------
# Plot the classification ROC curve
# Purpose: Train the structural classifier on the saved feature sets and export the ROC curve on the test split.
# -----------------------------------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

from plotting_standard import (
    apply_plot_style,
    print_key_value_rows,
    print_section,
    report_saved_paths,
    save_figure,
)

PROJECT_ROOT = Path(".").resolve()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
FIG_DIR = PROJECT_ROOT / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42

ALL_FEATURES_PATH = DATA_DIR / "features_dataset_with_shield.csv"
CLASS1_FEATURES_PATH = DATA_DIR / "features_dataset_with_shield_cluster1.csv"

AXIS_LABEL_SIZE = 17.5
TICK_LABEL_SIZE = 16.5
NOTE_SIZE = 16.5
BLUE = "#4C78A8"
LIGHT_GRAY = "#D9D9D9"


def build_classification_dataset():
    if not ALL_FEATURES_PATH.exists():
        raise FileNotFoundError(f"Full feature dataset not found: {ALL_FEATURES_PATH}")
    if not CLASS1_FEATURES_PATH.exists():
        raise FileNotFoundError(f"Cluster 1 feature dataset not found: {CLASS1_FEATURES_PATH}")

    df_all = pd.read_csv(ALL_FEATURES_PATH)
    df_class1 = pd.read_csv(CLASS1_FEATURES_PATH)

    candidate_keys = ["SMILES", "smiles", "ID", "id", "Name", "name"]
    match_key = None
    for key in candidate_keys:
        if key in df_all.columns and key in df_class1.columns:
            match_key = key
            break
    if match_key is None:
        raise ValueError("No shared matching key found between the full dataset and the cluster1 dataset.")

    df_all[match_key] = df_all[match_key].astype(str).str.strip()
    df_class1[match_key] = df_class1[match_key].astype(str).str.strip()
    class1_keys = set(df_class1[match_key].dropna().astype(str))
    df_all["target_class"] = df_all[match_key].astype(str).isin(class1_keys).astype(int)

    exclude_cols = {
        "target_class", "shift_value", match_key,
        "SMILES", "smiles", "Name", "name", "ID", "id",
        "cluster", "Cluster", "class", "Class", "priority_class", "reg_cluster",
    }
    feature_cols = [
        col for col in df_all.columns
        if col not in exclude_cols and pd.api.types.is_numeric_dtype(df_all[col])
    ]

    X = df_all[feature_cols].replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True)).fillna(0)
    y = df_all["target_class"].astype(int)
    return X, y


def train_classifier():
    X, y = build_classification_dataset()
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=SEED,
        stratify=y,
    )
    clf = ExtraTreesClassifier(
        n_estimators=500,
        max_features="sqrt",
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1,
    )
    clf.fit(X_train, y_train)
    return clf, X_test, y_test


apply_plot_style(profile="paper")
clf, X_test, y_test = train_classifier()
y_prob = clf.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc_value = float(roc_auc_score(y_test, y_prob))

print_section("Classification ROC summary")
print_key_value_rows([
    ("Test samples", len(y_test)),
    ("ROC-AUC", f"{auc_value:.3f}"),
])

fig, ax = plt.subplots(figsize=(5.2, 5.0), dpi=150)
ax.plot(
    fpr,
    tpr,
    color=BLUE,
    linewidth=2.2,
    label=f"ROC (AUC = {auc_value:.3f})",
)
ax.plot([0, 1], [0, 1], linestyle="--", color=LIGHT_GRAY, linewidth=1.4)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("False positive rate", fontsize=AXIS_LABEL_SIZE, labelpad=8)
ax.set_ylabel("True positive rate", fontsize=AXIS_LABEL_SIZE, labelpad=8)
ax.tick_params(axis="both", which="major", labelsize=TICK_LABEL_SIZE, pad=4)
ax.legend(loc="lower right", frameon=False, fontsize=NOTE_SIZE)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.8)

fig.tight_layout(pad=0.65)
saved_paths = save_figure(fig, FIG_DIR / "Figure_4b_classification_roc_curve.png")
plt.show()
report_saved_paths(saved_paths, "Saved figure files")


### Classification Feature Importance

Purpose: Rank and plot the structural features that drive the Class 0 versus Class 1 separation.


In [ ]:
# -----------------------------------------------------------------------------
# Plot classification feature importance
# Purpose: Rank the most important classifier features and export the feature-importance plot and table.
# -----------------------------------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import train_test_split

from plotting_standard import (
    apply_plot_style,
    print_key_value_rows,
    print_section,
    report_saved_paths,
    save_figure,
)

PROJECT_ROOT = Path(".").resolve()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
FIG_DIR = PROJECT_ROOT / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR = PROJECT_ROOT / "results" / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42

ALL_FEATURES_PATH = DATA_DIR / "features_dataset_with_shield.csv"
CLASS1_FEATURES_PATH = DATA_DIR / "features_dataset_with_shield_cluster1.csv"

AXIS_LABEL_SIZE = 17.5
TICK_LABEL_SIZE = 16.5
NOTE_SIZE = 16.5
MORGAN_BLUE = "#4C78A8"
ECFP_TEAL = "#2A9D8F"
MACCS_GRAY = "#9C9C9C"
SHIELD_ORANGE = "#F28E2B"
OTHER_GRAY = "#6E6E6E"


def build_classification_dataset():
    if not ALL_FEATURES_PATH.exists():
        raise FileNotFoundError(f"Full feature dataset not found: {ALL_FEATURES_PATH}")
    if not CLASS1_FEATURES_PATH.exists():
        raise FileNotFoundError(f"Cluster 1 feature dataset not found: {CLASS1_FEATURES_PATH}")

    df_all = pd.read_csv(ALL_FEATURES_PATH)
    df_class1 = pd.read_csv(CLASS1_FEATURES_PATH)

    candidate_keys = ["SMILES", "smiles", "ID", "id", "Name", "name"]
    match_key = None
    for key in candidate_keys:
        if key in df_all.columns and key in df_class1.columns:
            match_key = key
            break
    if match_key is None:
        raise ValueError("No shared matching key found between the full dataset and the cluster1 dataset.")

    df_all[match_key] = df_all[match_key].astype(str).str.strip()
    df_class1[match_key] = df_class1[match_key].astype(str).str.strip()
    class1_keys = set(df_class1[match_key].dropna().astype(str))
    df_all["target_class"] = df_all[match_key].astype(str).isin(class1_keys).astype(int)

    exclude_cols = {
        "target_class", "shift_value", match_key,
        "SMILES", "smiles", "Name", "name", "ID", "id",
        "cluster", "Cluster", "class", "Class", "priority_class", "reg_cluster",
    }
    feature_cols = [
        col for col in df_all.columns
        if col not in exclude_cols and pd.api.types.is_numeric_dtype(df_all[col])
    ]

    X = df_all[feature_cols].replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True)).fillna(0)
    y = df_all["target_class"].astype(int)
    return X, y, feature_cols


def train_classifier():
    X, y, feature_cols = build_classification_dataset()
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=SEED,
        stratify=y,
    )
    clf = ExtraTreesClassifier(
        n_estimators=500,
        max_features="sqrt",
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1,
    )
    clf.fit(X_train, y_train)
    return clf, feature_cols


def feature_color(feature_name):
    if feature_name.startswith("Morgan_"):
        return MORGAN_BLUE
    if feature_name.startswith("ECFP_F_"):
        return ECFP_TEAL
    if feature_name.startswith("MACCS_"):
        return MACCS_GRAY
    if feature_name == "shielding_value":
        return SHIELD_ORANGE
    return OTHER_GRAY


def display_label(feature_name):
    if feature_name == "shielding_value":
        return r"DFT shielding $\sigma$"
    return feature_name


apply_plot_style(profile="paper")
clf, feature_cols = train_classifier()

importance_df = (
    pd.DataFrame({"feature": feature_cols, "importance": clf.feature_importances_})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
top_df = importance_df.head(10).copy()
top_df["display_label"] = top_df["feature"].map(display_label)
top_df["color"] = top_df["feature"].map(feature_color)
plot_df = top_df.iloc[::-1].reset_index(drop=True)

csv_path = TABLE_DIR / "Figure_4c_classification_feature_importance_top10.csv"
top_df.to_csv(csv_path, index=False)
shielding_rank = int(importance_df.index[importance_df["feature"] == "shielding_value"][0]) + 1

print_section("Classification feature-importance summary")
print_key_value_rows([
    ("Total ranked features", len(importance_df)),
    ("Top feature", top_df.iloc[0]["feature"]),
    ("Shielding rank", shielding_rank),
])

fig, ax = plt.subplots(figsize=(8.6, 5.9), dpi=600)
bars = ax.barh(
    plot_df["display_label"],
    plot_df["importance"],
    color=plot_df["color"],
    edgecolor="black",
    linewidth=0.55,
    height=0.72,
)

x_max = float(plot_df["importance"].max())
ax.set_xlim(0, x_max * 1.23)

for bar, value in zip(bars, plot_df["importance"]):
    ax.text(
        value + x_max * 0.02,
        bar.get_y() + bar.get_height() / 2.0,
        f"{value:.3f}",
        ha="left",
        va="center",
        fontsize=NOTE_SIZE,
    )

ax.set_xlabel("Feature importance", fontsize=AXIS_LABEL_SIZE, labelpad=6)
ax.set_ylabel("")
ax.tick_params(axis="x", which="major", labelsize=TICK_LABEL_SIZE, pad=3)
ax.tick_params(axis="y", which="major", labelsize=TICK_LABEL_SIZE, pad=3)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.8)

fig.tight_layout(pad=0.7)
saved_paths = save_figure(fig, FIG_DIR / "Figure_4c_classification_feature_importance.png")
plt.show()
report_saved_paths([csv_path] + saved_paths, "Saved outputs")


### Representative Structural Motifs

Purpose: Render representative molecular motifs associated with the selected high-importance fingerprint bits.


In [ ]:
# -----------------------------------------------------------------------------
# Render representative structural motifs
# Purpose: Identify representative molecules for selected fingerprint bits and export the highlighted motif panel.
# -----------------------------------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors, Draw, rdDepictor, rdMolDescriptors

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import train_test_split

from plotting_standard import (
    apply_plot_style,
    print_key_value_rows,
    print_section,
    report_saved_paths,
    save_figure,
)

RDLogger.DisableLog("rdApp.*")

PROJECT_ROOT = Path(".").resolve()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
FIG_DIR = PROJECT_ROOT / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR = PROJECT_ROOT / "results" / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
N_BITS = 1024

ALL_FEATURES_PATH = DATA_DIR / "features_dataset_with_shield.csv"
CLASS1_FEATURES_PATH = DATA_DIR / "features_dataset_with_shield_cluster1.csv"

TITLE_SIZE = 22.5
NOTE_SIZE = 22.5
BORDER_COLOR = "#666666"
HIGHLIGHT_COLOR = (0.95, 0.55, 0.20)

SELECTED_BITS = [
    ("Morgan_32", 0, "Class 0-enriched"),
    ("ECFP_F_32", 0, "Class 0-enriched"),
    ("Morgan_26", 1, "Class 1-enriched"),
    ("ECFP_F_26", 1, "Class 1-enriched"),
    ("Morgan_429", 1, "Class 1-enriched"),
    ("ECFP_F_429", 1, "Class 1-enriched"),
]


def build_classification_frame():
    if not ALL_FEATURES_PATH.exists():
        raise FileNotFoundError(f"Full feature dataset not found: {ALL_FEATURES_PATH}")
    if not CLASS1_FEATURES_PATH.exists():
        raise FileNotFoundError(f"Cluster 1 feature dataset not found: {CLASS1_FEATURES_PATH}")

    df_all = pd.read_csv(ALL_FEATURES_PATH)
    df_class1 = pd.read_csv(CLASS1_FEATURES_PATH)

    candidate_keys = ["SMILES", "smiles", "ID", "id", "Name", "name"]
    match_key = None
    for key in candidate_keys:
        if key in df_all.columns and key in df_class1.columns:
            match_key = key
            break
    if match_key is None:
        raise ValueError("No shared matching key found between the full dataset and the cluster1 dataset.")

    df_all[match_key] = df_all[match_key].astype(str).str.strip()
    df_class1[match_key] = df_class1[match_key].astype(str).str.strip()
    class1_keys = set(df_class1[match_key].dropna().astype(str))
    df_all["target_class"] = df_all[match_key].astype(str).isin(class1_keys).astype(int)
    return df_all


def build_training_matrices(df_all):
    exclude_cols = {
        "target_class", "shift_value",
        "SMILES", "smiles", "Name", "name", "ID", "id",
        "cluster", "Cluster", "class", "Class", "priority_class", "reg_cluster",
    }
    feature_cols = [
        col for col in df_all.columns
        if col not in exclude_cols and pd.api.types.is_numeric_dtype(df_all[col])
    ]
    X = df_all[feature_cols].replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True)).fillna(0)
    y = df_all["target_class"].astype(int)
    return X, y, feature_cols


def train_classifier(df_all):
    X, y, feature_cols = build_training_matrices(df_all)
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=SEED,
        stratify=y,
    )
    clf = ExtraTreesClassifier(
        n_estimators=500,
        max_features="sqrt",
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1,
    )
    clf.fit(X_train, y_train)
    return clf, feature_cols


def build_env_from_center_radius(mol, center_atom, radius):
    env_bonds = Chem.FindAtomEnvironmentOfRadiusN(mol, radius, center_atom)
    atoms = {center_atom}
    for bond_idx in env_bonds:
        bond = mol.GetBondWithIdx(bond_idx)
        atoms.update([bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()])
    return sorted(atoms), sorted(env_bonds)


def get_morgan_highlight(smiles, bit_id, radius=2):
    mol = Chem.MolFromSmiles(smiles)
    bit_info = {}
    AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius,
        nBits=N_BITS,
        bitInfo=bit_info,
        useChirality=False,
    )
    if bit_id not in bit_info:
        return mol, [], []

    fluorines = {atom.GetIdx() for atom in mol.GetAtoms() if atom.GetAtomicNum() == 9}
    chosen = bit_info[bit_id][0]
    for center, bit_radius in bit_info[bit_id]:
        atoms, _ = build_env_from_center_radius(mol, center, bit_radius)
        if set(atoms) & fluorines:
            chosen = (center, bit_radius)
            break
    atoms, bonds = build_env_from_center_radius(mol, chosen[0], chosen[1])
    return mol, atoms, bonds


def get_ecfp_f_highlight(smiles, bit_id, radius=4):
    mol = Chem.MolFromSmiles(smiles)
    fluorines = [atom.GetIdx() for atom in mol.GetAtoms() if atom.GetAtomicNum() == 9]
    for fluorine_idx in fluorines:
        bit_info = {}
        AllChem.GetMorganFingerprintAsBitVect(
            mol,
            radius,
            nBits=N_BITS,
            bitInfo=bit_info,
            useChirality=True,
            fromAtoms=[fluorine_idx],
        )
        if bit_id not in bit_info:
            continue
        center_atom, bit_radius = bit_info[bit_id][0]
        atoms, bonds = build_env_from_center_radius(mol, center_atom, bit_radius)
        atoms = sorted(set(atoms) | {fluorine_idx})
        return mol, atoms, bonds
    return mol, [], []


def get_highlight_for_bit(smiles, bit_name):
    bit_id = int(bit_name.split("_")[-1])
    if bit_name.startswith("Morgan_"):
        return get_morgan_highlight(smiles, bit_id)
    return get_ecfp_f_highlight(smiles, bit_id)


def build_structure_profile(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    fluorine_atoms = [atom for atom in mol.GetAtoms() if atom.GetAtomicNum() == 9]
    ring_bound_f = 0
    aromatic_f = 0
    hetero_adj_f = 0
    for atom in fluorine_atoms:
        neighbor = atom.GetNeighbors()[0] if atom.GetDegree() else None
        if neighbor is None:
            continue
        ring_bound_f += int(neighbor.IsInRing())
        aromatic_f += int(neighbor.GetIsAromatic())
        hetero_adj_f += int(
            any(
                nbr.GetAtomicNum() not in (1, 6, 9)
                for nbr in neighbor.GetNeighbors()
                if nbr.GetIdx() != atom.GetIdx()
            )
        )

    hetero_count = sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() not in (1, 6, 9))
    return {
        "f_count": len(fluorine_atoms),
        "ring_bound_f": ring_bound_f,
        "aromatic_f": aromatic_f,
        "hetero_adj_f": hetero_adj_f,
        "hetero_count": hetero_count,
        "ring_count": int(rdMolDescriptors.CalcNumRings(mol)),
        "heavy_atoms": mol.GetNumHeavyAtoms(),
        "fragment_count": smiles.count(".") + 1,
        "molecular_weight": float(Descriptors.MolWt(mol)),
    }


def representative_score(bit_name, target_class, profile, already_used=False):
    score = 0.0
    if target_class == 0:
        score += 12 if profile["f_count"] == 1 else -8 * abs(profile["f_count"] - 1)
        score += 8 if profile["ring_bound_f"] >= 1 else -6
        score += 4 if profile["aromatic_f"] == 0 else -8
        score += 3 if profile["hetero_adj_f"] == 0 else -2 * profile["hetero_adj_f"]
        score -= 0.9 * abs(profile["hetero_count"] - 2)
        score -= 0.45 * abs(profile["heavy_atoms"] - 14)
        score -= 0.8 * abs(profile["ring_count"] - 2)
    elif bit_name.endswith("_26"):
        score += 12 if profile["f_count"] == 2 else -8 * abs(profile["f_count"] - 2)
        score += 6 if profile["ring_bound_f"] == 0 else -3 * profile["ring_bound_f"]
        score += 3 if profile["aromatic_f"] == 0 else -4
        score += 3 if profile["hetero_count"] >= 2 else -4
        score += 2 if profile["ring_count"] >= 1 else -3
        score -= 0.35 * abs(profile["heavy_atoms"] - 15)
        score -= 0.5 * abs(profile["hetero_count"] - 3)
    else:
        score += 12 if profile["f_count"] == 3 else -8 * abs(profile["f_count"] - 3)
        score += 6 if profile["ring_bound_f"] == 0 else -4 * profile["ring_bound_f"]
        score += 4 if profile["hetero_count"] >= 3 else -3
        score += 2 if profile["ring_count"] >= 1 else -4
        score += 1.5 if profile["heavy_atoms"] >= 12 else -4
        score -= 0.25 * abs(profile["heavy_atoms"] - 16)
        score -= 0.5 * abs(profile["hetero_count"] - 4)

    if profile["fragment_count"] > 1:
        score -= 4
    if already_used:
        score -= 6
    return float(score)


def choose_representative(df_all, bit_name, target_class, used_smiles=None):
    if used_smiles is None:
        used_smiles = set()
    subset = df_all.loc[
        (df_all["target_class"] == target_class) & (df_all[bit_name] == 1),
        ["SMILES", "shift_value", "shielding_value", "target_class", bit_name],
    ].copy()
    subset["SMILES"] = subset["SMILES"].astype(str).str.strip()
    subset["shift_value"] = pd.to_numeric(subset["shift_value"], errors="coerce")
    subset["shielding_value"] = pd.to_numeric(subset["shielding_value"], errors="coerce")
    subset = subset.dropna(subset=["SMILES", "shift_value", "shielding_value"]).drop_duplicates(subset=["SMILES"])
    if subset.empty:
        raise ValueError(f"No representative molecules found for {bit_name} in class {target_class}.")

    scored_rows = []
    for _, row in subset.iterrows():
        profile = build_structure_profile(row["SMILES"])
        if profile is None:
            continue
        scored_rows.append({
            **row.to_dict(),
            **profile,
            "selection_score": representative_score(
                bit_name,
                target_class,
                profile,
                already_used=row["SMILES"] in used_smiles,
            ),
        })

    if not scored_rows:
        raise ValueError(f"Unable to find a parseable representative for {bit_name}.")

    scored_df = pd.DataFrame(scored_rows).sort_values(
        ["selection_score", "fragment_count", "heavy_atoms", "hetero_count", "SMILES"],
        ascending=[False, True, True, True, True],
    ).reset_index(drop=True)
    return scored_df.iloc[0]


def render_highlighted_molecule(smiles, bit_name):
    mol, atoms, bonds = get_highlight_for_bit(smiles, bit_name)

    if mol is None:
        raise ValueError(f"RDKit failed to parse representative SMILES: {smiles}")

    mol = Chem.Mol(mol)
    rdDepictor.Compute2DCoords(mol)
    return Draw.MolToImage(
        mol,
        size=(500, 320),
        highlightAtoms=atoms,
        highlightBonds=bonds,
        highlightColor=HIGHLIGHT_COLOR,
        fitImage=True,
    )


apply_plot_style(profile="paper")
df_all = build_classification_frame()
clf, feature_cols = train_classifier(df_all)
importance_df = (
    pd.DataFrame({"feature": feature_cols, "importance": clf.feature_importances_})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

records = []
used_smiles = set()
fig, axes = plt.subplots(2, 3, figsize=(15.2, 8.8), dpi=600)
axes = axes.flatten()

for ax, (bit_name, target_class, note) in zip(axes, SELECTED_BITS):
    row = choose_representative(df_all, bit_name, target_class, used_smiles)
    used_smiles.add(row["SMILES"])
    image = render_highlighted_molecule(row["SMILES"], bit_name)
    importance_rank = int(importance_df.index[importance_df["feature"] == bit_name][0]) + 1

    ax.imshow(image)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.8)
        spine.set_edgecolor(BORDER_COLOR)

    ax.set_title(f"{bit_name}\n({note})", fontsize=TITLE_SIZE, pad=10)

    records.append({
        "bit_name": bit_name,
        "target_class": int(target_class),
        "enrichment_note": note,
        "importance_rank": importance_rank,
        "selection_score": float(row["selection_score"]),
        "SMILES": row["SMILES"],
        "shift_value": float(row["shift_value"]),
        "shielding_value": float(row["shielding_value"]),
        "f_count": int(row["f_count"]),
        "ring_bound_f": int(row["ring_bound_f"]),
        "aromatic_f": int(row["aromatic_f"]),
        "hetero_adj_f": int(row["hetero_adj_f"]),
        "hetero_count": int(row["hetero_count"]),
        "ring_count": int(row["ring_count"]),
        "heavy_atoms": int(row["heavy_atoms"]),
        "fragment_count": int(row["fragment_count"]),
        "molecular_weight": float(row["molecular_weight"]),
    })

csv_path = TABLE_DIR / "Figure_4d_classification_representative_motifs.csv"
pd.DataFrame(records).to_csv(csv_path, index=False)

print_section("Representative motif summary")
print_key_value_rows([
    ("Representative motifs", len(records)),
    ("Saved selection table", csv_path),
])

fig.subplots_adjust(left=0.03, right=0.99, top=0.91, bottom=0.05, hspace=0.18, wspace=0.10)
saved_paths = save_figure(fig, FIG_DIR / "Figure_4d_classification_representative_motifs.png")
plt.show()
report_saved_paths([csv_path] + saved_paths, "Saved outputs")


### Shielding-Versus-Shift Relationship

Purpose: Plot the relationship between calculated isotropic shielding values and experimental chemical shifts.


In [ ]:
# -----------------------------------------------------------------------------
# Plot the shielding-versus-shift relationship
# Purpose: Load the shielding-aware cleaned dataset and export the shielding-versus-shift plot.
# -----------------------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------- Input ----------
from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = Path('.').resolve()

if 'df_cleaned' not in globals():
    CLEANED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'cleaned_shift_data_with_shield.csv'
    if not CLEANED_PATH.exists():
        raise FileNotFoundError(
            'df_cleaned not found and cleaned file not found. '
            'Please run the preprocessing cell first or ensure '
            'cleaned_shift_data_with_shield.csv exists.'
        )
    df_cleaned = pd.read_csv(CLEANED_PATH)

required_cols = {'SMILES', 'shift_value', 'shielding_value'}
missing = required_cols - set(df_cleaned.columns)
if missing:
    raise ValueError(f'df_cleaned missing required columns: {missing}')

# ---------- Style ----------
apply_plot_style(profile='compact')

# ---------- Data ----------
df_cleaned['shift_value'] = pd.to_numeric(df_cleaned['shift_value'], errors='coerce')
df_cleaned['shielding_value'] = pd.to_numeric(df_cleaned['shielding_value'], errors='coerce')
df_plot = df_cleaned.dropna(subset=['shift_value', 'shielding_value']).copy()

RESULTS_DIR = PROJECT_ROOT / 'results'
FIG_DIR = RESULTS_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

OUT_FIG = FIG_DIR / 'Figure_5a_shielding_vs_shift_final.png'

print_section('Shielding-versus-shift dataset overview')
print_key_value_rows([
    ('Samples used', len(df_plot)),
    ('Shift range / ppm', f"{df_plot['shift_value'].min():.3f} to {df_plot['shift_value'].max():.3f}"),
    ('Shielding range', f"{df_plot['shielding_value'].min():.6f} to {df_plot['shielding_value'].max():.6f}"),
])

# ---------- Plot ----------
fig, ax = create_single_axis_figure('single_wide')

x = df_plot['shielding_value'].to_numpy()
y = df_plot['shift_value'].to_numpy()

# main scatter
ax.scatter(
    x,
    y,
    s=10,
    alpha=0.70,
    color='#4C78A8',
    edgecolors='none',
    rasterized=True
)



# labels
ax.set_xlabel(r'Isotropic shielding constant, $\sigma$')
ax.set_ylabel(r'Experimental $^{19}$F chemical shift, $\delta_{\mathrm{exp}}$ (ppm)')

# full box
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.7)

ax.tick_params(axis='both', which='major', pad=1.5)

# annotation
ax.text(
    0.97, 0.96,
    f'n = {len(df_plot)}',
    transform=ax.transAxes,
    ha='right',
    va='top',
    fontsize=7
)

fig.tight_layout(pad=0.35)

saved_paths = save_figure(fig, OUT_FIG)
plt.show()
report_saved_paths(saved_paths, 'Saved figure files')


### Cluster 1 Priority-Band Assignment

Purpose: Visualize the band-based assignment around the selected Cluster 1 regression line.


In [ ]:
# -----------------------------------------------------------------------------
# Visualize the Cluster 1 priority-band assignment
# Purpose: Load the saved priority-band assignment results and export the cluster-assignment plot.
# -----------------------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

PROJECT_ROOT = Path('.').resolve()
RESULTS_DIR = PROJECT_ROOT / 'results'
FIG_DIR = RESULTS_DIR / 'figures'
TAB_DIR = RESULTS_DIR / 'tables'
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

# ---------- Unified colors ----------
BLUE = '#4C78A8'
ORANGE = '#F28E2B'
LIGHT_GRAY = '#D9D9D9'

# ---------- Style ----------
apply_plot_style(profile='compact')

if 'df_out' not in globals():
    raise RuntimeError('df_out not found. Please run the regression-mixture assignment cell first.')
if 'models' not in globals():
    raise RuntimeError('models not found. Please run the regression-mixture assignment cell first.')

# Internal model index retained for compatibility
TARGET_MODEL_ID = 1
TARGET_MODEL_LABEL_COL = 'reg_cluster'

mask_target = df_out[TARGET_MODEL_LABEL_COL] == TARGET_MODEL_ID
resid_target = df_out.loc[mask_target, 'shift_value'].values - models[TARGET_MODEL_ID].predict(
    df_out.loc[mask_target, 'shielding_value'].values.reshape(-1, 1)
)

SD_res = float(np.std(resid_target, ddof=1))
K_SD = 2
thr = K_SD * SD_res

print_section('Band-assignment settings')
print_key_value_rows([
    ('Selected model ID', TARGET_MODEL_ID),
    ('Residual standard deviation (SD_res)', f'{SD_res:.3f}'),
    ('Band width', f'±{thr:.3f} (= ±{K_SD}·SD_res)'),
])

x_all = df_out['shielding_value'].values.reshape(-1, 1)
y_all = df_out['shift_value'].values
yhat_target = models[TARGET_MODEL_ID].predict(x_all)
resid_all_target = np.abs(y_all - yhat_target)

df_class = df_out.copy()
df_class['priority_class'] = np.where(resid_all_target <= thr, 1, 0)
df_class['residual_to_reference_line'] = resid_all_target

n1 = int((df_class['priority_class'] == 1).sum())
n0 = int((df_class['priority_class'] == 0).sum())

print_key_value_rows([
    ('Class 1 count', f'{n1} ({n1 / len(df_class) * 100:.2f}%)'),
    ('Class 0 count', f'{n0} ({n0 / len(df_class) * 100:.2f}%)'),
])

# ---------- Save table ----------
out_csv_name = f'band_based_class_assignment_k{str(K_SD).replace(".", "p")}.csv'
OUT_CSV = TAB_DIR / out_csv_name
df_class.to_csv(OUT_CSV, index=False)
report_saved_paths([OUT_CSV], 'Saved class-assignment table')

# ---------- Plot ----------
OUT_FIG = FIG_DIR / f'Figure_5_band_based_class_assignment_k{str(K_SD).replace(".", "p")}.png'

fig, ax = create_single_axis_figure('single_wide')

mask1 = df_class['priority_class'] == 1
mask0 = ~mask1

x_line = np.linspace(
    df_class['shielding_value'].min(),
    df_class['shielding_value'].max(),
    300
).reshape(-1, 1)
y_line = models[TARGET_MODEL_ID].predict(x_line)

# band first
band = ax.fill_between(
    x_line.ravel(),
    y_line - thr,
    y_line + thr,
    color=LIGHT_GRAY,
    alpha=0.35,
    linewidth=0.0,
    label=rf'$\pm {K_SD}\cdot \mathrm{{SD}}_{{\mathrm{{res}}}}$',
    zorder=0
)

# Class 0
sc0 = ax.scatter(
    df_class.loc[mask0, 'shielding_value'],
    df_class.loc[mask0, 'shift_value'],
    s=10,
    alpha=0.62,
    color=ORANGE,
    edgecolors='none',
    label='Class 0',
    rasterized=True,
    zorder=1
)

# Class 1
sc1 = ax.scatter(
    df_class.loc[mask1, 'shielding_value'],
    df_class.loc[mask1, 'shift_value'],
    s=10,
    alpha=0.78,
    color=BLUE,
    edgecolors='none',
    label='Class 1',
    rasterized=True,
    zorder=2
)

# reference line
ln, = ax.plot(
    x_line.ravel(),
    y_line,
    color='black',
    linewidth=1.0,
    label='Reference line',
    zorder=3
)

# labels
ax.set_xlabel(r'Isotropic shielding constant, $\sigma$')
ax.set_ylabel(r'Experimental $^{19}$F chemical shift, $\delta_{\mathrm{exp}}$ (ppm)')

# full box
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.7)

ax.tick_params(axis='both', which='major', pad=1.5)

# annotation
ax.text(
    0.95, 0.95,
    f'Class 1: {n1}\nClass 0: {n0}',
    transform=ax.transAxes,
    ha='right',
    va='top',
    fontsize=6.8
)

# legend
ax.legend(
    handles=[sc1, sc0, ln, band],
    labels=[
        'Class 1',
        'Class 0',
        'Reference line',
        rf'$\pm {K_SD}\cdot \mathrm{{SD}}_{{\mathrm{{res}}}}$'
    ],
    loc='lower left',
    frameon=False,
    handlelength=1.6,
    borderpad=0.2,
    labelspacing=0.3
)

fig.tight_layout(pad=0.35)

saved_paths = save_figure(fig, OUT_FIG)
plt.show()

report_saved_paths(saved_paths, 'Saved figure files')


### ExtraTrees Comparison With and Without Shielding

Purpose: Compare the structure-only and shielding-aware ExtraTrees models on the test split.


In [ ]:
# -----------------------------------------------------------------------------
# Compare ExtraTrees models with and without shielding
# Purpose: Load the saved refined ExtraTrees models, evaluate them on the aligned test split, and export the comparison plot.
# -----------------------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

SEED = 42

FEATURE_NO_SIGMA = Path('data/processed/features_dataset.parquet')
FEATURE_WITH_SIGMA = Path('data/processed/features_dataset_with_shield.parquet')

MODEL_NO_SIGMA = Path('results/models/best_on_val_refined_ExtraTrees.pkl')
MODEL_WITH_SIGMA = Path('results/models/best_on_val_refined_ExtraTrees_with_shield.pkl')

OUT_PNG = Path('results/figures/Figure4_ExtraTrees_Test_AB.png')
OUT_PNG.parent.mkdir(parents=True, exist_ok=True)

apply_plot_style()

def regression_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

def load_xy_predict(feature_path, model_path):
    df = pd.read_parquet(feature_path)
    X = df.drop(columns=['SMILES', 'shift_value'])
    y = df['shift_value'].astype(float)

    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=0.10, random_state=SEED
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=(1 / 9), random_state=SEED
    )
    model = joblib.load(model_path)
    y_pred_test = model.predict(X_test)

    mae, rmse, r2 = regression_metrics(y_test, y_pred_test)
    return y_test.values, y_pred_test, (mae, rmse, r2)

def parity(ax, y_true, y_pred, title, metrics, vmin, vmax):
    mae, rmse, r2 = metrics
    ax.scatter(y_true, y_pred, s=18, alpha=0.80, edgecolor='none')
    add_identity_line(ax, vmin, vmax)

    ax.set_xlim(vmin, vmax)
    ax.set_ylim(vmin, vmax)
    ax.set_title(title)
    ax.set_xlabel('True shift (ppm)')
    ax.set_ylabel('Predicted shift (ppm)')
    style_axes(ax, equal_aspect=True)

    annotate_panel_text(
        ax,
        f'MAE={mae:.2f} ppm\nRMSE={rmse:.2f} ppm\nR$^2$={r2:.3f}',
    )

y_true_A, y_pred_A, mA = load_xy_predict(FEATURE_NO_SIGMA, MODEL_NO_SIGMA)
y_true_B, y_pred_B, mB = load_xy_predict(FEATURE_WITH_SIGMA, MODEL_WITH_SIGMA)

vmin, vmax = compute_plot_limits(y_true_A, y_pred_A, y_true_B, y_pred_B)

fig, axes = create_figure('comparison', nrows=1, ncols=2, constrained_layout=True)

parity(axes[0], y_true_A, y_pred_A, 'ExtraTrees (structural only) - Test', mA, vmin, vmax)
parity(axes[1], y_true_B, y_pred_B, 'ExtraTrees (structural + shielding) - Test', mB, vmin, vmax)

saved_paths = save_figure(fig, OUT_PNG)
plt.show()

report_saved_paths(saved_paths, 'Saved figure files')


### ExtraTrees Comparison Before and After Cluster 1 Cleaning

Purpose: Compare ExtraTrees performance before and after applying the Cluster 1 cleaning workflow.


In [ ]:
# -----------------------------------------------------------------------------
# Compare ExtraTrees performance before and after Cluster 1 cleaning
# Purpose: Load the saved refined ExtraTrees models for the full and cleaned datasets and export the comparison plot.
# -----------------------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

SEED = 42
FEATURE_BEFORE = Path('data/processed/features_dataset_with_shield.parquet')
FEATURE_AFTER = Path('data/processed/features_dataset_with_shield_cluster1.parquet')
MODEL_BEFORE = Path('results/models/best_on_val_refined_ExtraTrees_with_shield.pkl')
MODEL_AFTER = Path('results/models/best_on_val_refined_ExtraTrees_cluster1.pkl')

OUT_PNG = Path('results/figures/Figure4_ExtraTrees_Cleaning_Test_AB.png')
OUT_PNG.parent.mkdir(parents=True, exist_ok=True)

apply_plot_style()

def regression_metrics(y_true, y_pred):
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred))
    return mae, rmse, r2

def load_xy_predict(feature_path, model_path):
    df = pd.read_parquet(feature_path)
    X = df.drop(columns=['SMILES', 'shift_value'])
    y = df['shift_value'].astype(float)
    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.10, random_state=SEED)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=(1 / 9), random_state=SEED)

    loaded = joblib.load(model_path)
    if isinstance(loaded, dict):
        for k in ['model', 'best_model', 'best_estimator', 'best_estimator_', 'estimator']:
            if k in loaded:
                loaded = loaded[k]
                break
        else:
            raise TypeError(f'Loaded object is a dict but no known model key found. Available keys: {list(loaded.keys())}')

    y_pred_test = loaded.predict(X_test)
    mae, rmse, r2 = regression_metrics(y_test, y_pred_test)
    return y_test.values, y_pred_test, (mae, rmse, r2)

def parity(ax, y_true, y_pred, title, metrics, vmin, vmax):
    mae, rmse, r2 = metrics
    ax.scatter(y_true, y_pred, s=18, alpha=0.80, edgecolor='none')
    add_identity_line(ax, vmin, vmax)
    ax.set_xlim(vmin, vmax)
    ax.set_ylim(vmin, vmax)
    ax.set_title(title)
    ax.set_xlabel('True shift (ppm)')
    ax.set_ylabel('Predicted shift (ppm)')
    style_axes(ax, equal_aspect=True)
    annotate_panel_text(ax, metric_text(mae, rmse, r2, decimals=2))

y_true_A, y_pred_A, mA = load_xy_predict(FEATURE_BEFORE, MODEL_BEFORE)
y_true_B, y_pred_B, mB = load_xy_predict(FEATURE_AFTER, MODEL_AFTER)

vmin, vmax = compute_plot_limits(y_true_A, y_pred_A, y_true_B, y_pred_B)
fig, axes = create_figure('comparison', nrows=1, ncols=2, constrained_layout=True)
parity(axes[0], y_true_A, y_pred_A, 'ExtraTrees with shielding (test)', mA, vmin, vmax)
parity(axes[1], y_true_B, y_pred_B, 'ExtraTrees with shielding (Cluster 1 test)', mB, vmin, vmax)

saved_paths = save_figure(fig, OUT_PNG)
plt.show()

report_saved_paths(saved_paths, 'Saved figure files')


### ExtraTrees-Versus-GCN Comparison

Purpose: Compare the shielding-aware ExtraTrees model against the shielding-aware GCN on the test split.


In [ ]:
# -----------------------------------------------------------------------------
# Compare ExtraTrees and GCN on the test split
# Purpose: Load the saved tabular and graph models, evaluate them on the test split, and export the parity comparison.
# -----------------------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, BatchNorm, global_mean_pool

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print_section('Execution environment')
print_key_value_rows([('Device', DEVICE)])

apply_plot_style()
PROJECT_ROOT = Path('.').resolve()
FEATURE_WITH_SIGMA = PROJECT_ROOT / 'data' / 'processed' / 'features_dataset_with_shield.parquet'
MODEL_WITH_SIGMA = PROJECT_ROOT / 'results' / 'models' / 'best_on_val_refined_ExtraTrees_with_shield.pkl'
GCN_TEST_LOADER_PTH = PROJECT_ROOT / 'results' / 'gcn' / 'test_loader.pth'
GCN_CKPT_PATH = PROJECT_ROOT / 'results' / 'gcn' / 'gcn_best_with_shield.pt'

OUT_DIR = PROJECT_ROOT / 'results' / 'figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PNG = OUT_DIR / 'Figure5_ExtraTrees_vs_GCN_Test_AB.png'

def regression_metrics(y_true, y_pred):
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred))
    return mae, rmse, r2

def plain_split_tabular(X, y, seed=42):
    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.10, random_state=seed)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=(1 / 9), random_state=seed)
    return X_train, X_val, X_test, y_train, y_val, y_test

def load_xy_predict_extratrees(feature_path, model_path, seed=42):
    df = pd.read_parquet(feature_path)
    required_cols = {'SMILES', 'shielding_value', 'shift_value'}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f'Parquet missing required columns: {missing}. Found: {list(df.columns)}')

    X = df.drop(columns=['shift_value'])
    y = df['shift_value'].astype(float)
    X_train, X_val, X_test, y_train, y_val, y_test = plain_split_tabular(X, y, seed=seed)
    X_test_tab = X_test.drop(columns=['SMILES']) if 'SMILES' in X_test.columns else X_test

    model = joblib.load(model_path)
    if isinstance(model, dict) and 'model' in model:
        model = model['model']

    y_pred_test = model.predict(X_test_tab)
    return y_test.values, y_pred_test, regression_metrics(y_test, y_pred_test)

class GCN(torch.nn.Module):
    def __init__(self, num_node_features, hidden_channels=512, num_layers=3, predictor_hidden_feats=1024, dropout=0.08343724004088336, use_residual=True, use_global_shielding=True, global_feat_dim=1):
        super().__init__()
        self.dropout = dropout
        self.use_residual = use_residual
        self.num_layers = num_layers
        self.use_global_shielding = use_global_shielding

        self.convs = torch.nn.ModuleList()
        self.bns = torch.nn.ModuleList()
        self.convs.append(GCNConv(num_node_features, hidden_channels))
        self.bns.append(BatchNorm(hidden_channels))
        for _ in range(1, num_layers):
            self.convs.append(GCNConv(hidden_channels, hidden_channels))
            self.bns.append(BatchNorm(hidden_channels))

        pred_in_dim = hidden_channels + (global_feat_dim if use_global_shielding else 0)
        self.fc1 = torch.nn.Linear(pred_in_dim, predictor_hidden_feats)
        self.fc2 = torch.nn.Linear(predictor_hidden_feats, 1)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x_res = None
        for i in range(self.num_layers):
            x = self.convs[i](x, edge_index)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            if self.use_residual and x_res is not None and x_res.shape[-1] == x.shape[-1]:
                x = x + x_res
            x_res = x

        g = global_mean_pool(x, batch)
        if self.use_global_shielding:
            shielding = data.shielding.view(-1, 1).to(g.dtype)
            g = torch.cat([g, shielding], dim=1)

        g = F.relu(self.fc1(g))
        g = F.dropout(g, p=self.dropout, training=self.training)
        out = self.fc2(g)
        return out.squeeze()

def load_gcn_from_ckpt(ckpt_path, num_node_features, device):
    ckpt = torch.load(ckpt_path, map_location=device)
    best_params = ckpt.get('best_params', {})
    model = GCN(
        num_node_features=num_node_features,
        hidden_channels=int(best_params.get('hidden_channels', 512)),
        num_layers=int(best_params.get('num_layers', 3)),
        predictor_hidden_feats=int(best_params.get('predictor_hidden_feats', 1024)),
        dropout=float(best_params.get('dropout', 0.08343724004088336)),
        use_residual=True,
        use_global_shielding=True,
        global_feat_dim=1,
    ).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return model

def parity(ax, y_true, y_pred, title, metrics, vmin, vmax):
    mae, rmse, r2 = metrics
    ax.scatter(y_true, y_pred, s=18, alpha=0.80, edgecolor='none')
    add_identity_line(ax, vmin, vmax)
    ax.set_xlim(vmin, vmax)
    ax.set_ylim(vmin, vmax)
    ax.set_title(title)
    ax.set_xlabel('True shift (ppm)')
    ax.set_ylabel('Predicted shift (ppm)')
    style_axes(ax, equal_aspect=True)
    annotate_panel_text(ax, metric_text(mae, rmse, r2, decimals=2))

y_true_et, y_pred_et, m_et = load_xy_predict_extratrees(FEATURE_WITH_SIGMA, MODEL_WITH_SIGMA, seed=SEED)
print('ExtraTrees + shielding (test split from parquet) metrics:', m_et)

test_loader = torch.load(GCN_TEST_LOADER_PTH, map_location=DEVICE)
first_batch = next(iter(test_loader))
num_node_features = first_batch.x.shape[1]
print('GCN num_node_features (from test_loader):', num_node_features)

gcn = load_gcn_from_ckpt(GCN_CKPT_PATH, num_node_features=num_node_features, device=DEVICE)
trues, preds = [], []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(DEVICE)
        pred = gcn(batch).detach().cpu().numpy()
        ytrue = batch.y.view(-1).detach().cpu().numpy()
        preds.append(pred)
        trues.append(ytrue)

y_true_gcn = np.concatenate(trues)
y_pred_gcn = np.concatenate(preds)
m_gcn = regression_metrics(y_true_gcn, y_pred_gcn)
print(metric_row('GCN', *m_gcn))

vmin, vmax = compute_plot_limits(y_true_et, y_pred_et, y_true_gcn, y_pred_gcn)
fig, axes = create_figure('comparison', nrows=1, ncols=2, constrained_layout=True)
parity(axes[0], y_true_et, y_pred_et, 'ExtraTrees with shielding (test)', m_et, vmin, vmax)
parity(axes[1], y_true_gcn, y_pred_gcn, 'GCN with shielding (test)', m_gcn, vmin, vmax)

saved_paths = save_figure(fig, OUT_PNG)
plt.show()

report_saved_paths(saved_paths, 'Saved figure files')


### GCN Comparison Before and After Cluster 1 Cleaning

Purpose: Compare GCN performance before and after applying the Cluster 1 cleaning workflow.


In [ ]:
# -----------------------------------------------------------------------------
# Compare GCN performance before and after Cluster 1 cleaning
# Purpose: Load the saved GCN checkpoints before and after cleaning, evaluate them, and export the comparison plot.
# -----------------------------------------------------------------------------

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, BatchNorm, global_mean_pool

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print_section('Execution environment')
print_key_value_rows([('Device', DEVICE)])

PROJECT_ROOT = Path('.').resolve()
apply_plot_style()

GCN_TEST_LOADER_BEFORE = PROJECT_ROOT / 'results' / 'gcn' / 'test_loader.pth'
GCN_CKPT_BEFORE = PROJECT_ROOT / 'results' / 'gcn' / 'gcn_best_with_shield.pt'
GCN_TEST_LOADER_AFTER = PROJECT_ROOT / 'results' / 'gcn' / 'test_loader_cluster1.pth'
GCN_CKPT_AFTER = PROJECT_ROOT / 'results' / 'gcn' / 'gcn_best_with_shield_cluster1.pt'

OUT_DIR = PROJECT_ROOT / 'results' / 'figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PNG = OUT_DIR / 'Figure5_GCN_Cleaning_Test_AB.png'

def regression_metrics(y_true, y_pred):
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred))
    return mae, rmse, r2

class GCN(torch.nn.Module):
    def __init__(self, num_node_features, hidden_channels=512, num_layers=3, predictor_hidden_feats=1024, dropout=0.08343724004088336, use_residual=True, use_global_shielding=True, global_feat_dim=1):
        super().__init__()
        self.dropout = dropout
        self.use_residual = use_residual
        self.num_layers = num_layers
        self.use_global_shielding = use_global_shielding

        self.convs = torch.nn.ModuleList()
        self.bns = torch.nn.ModuleList()
        self.convs.append(GCNConv(num_node_features, hidden_channels))
        self.bns.append(BatchNorm(hidden_channels))
        for _ in range(1, num_layers):
            self.convs.append(GCNConv(hidden_channels, hidden_channels))
            self.bns.append(BatchNorm(hidden_channels))

        pred_in_dim = hidden_channels + (global_feat_dim if use_global_shielding else 0)
        self.fc1 = torch.nn.Linear(pred_in_dim, predictor_hidden_feats)
        self.fc2 = torch.nn.Linear(predictor_hidden_feats, 1)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x_res = None
        for i in range(self.num_layers):
            x = self.convs[i](x, edge_index)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            if self.use_residual and x_res is not None and x_res.shape[-1] == x.shape[-1]:
                x = x + x_res
            x_res = x

        g = global_mean_pool(x, batch)
        if self.use_global_shielding:
            if not hasattr(data, 'shielding'):
                raise AttributeError('Data object missing shielding while use_global_shielding=True')
            shielding = data.shielding.view(-1, 1).to(g.dtype)
            g = torch.cat([g, shielding], dim=1)

        g = F.relu(self.fc1(g))
        g = F.dropout(g, p=self.dropout, training=self.training)
        out = self.fc2(g)
        return out.squeeze()

def load_gcn_from_ckpt(ckpt_path, num_node_features, device):
    ckpt = torch.load(ckpt_path, map_location=device)
    best_params = ckpt.get('best_params', {})
    model = GCN(
        num_node_features=num_node_features,
        hidden_channels=int(best_params.get('hidden_channels', 512)),
        num_layers=int(best_params.get('num_layers', 3)),
        predictor_hidden_feats=int(best_params.get('predictor_hidden_feats', 1024)),
        dropout=float(best_params.get('dropout', 0.08343724004088336)),
        use_residual=True,
        use_global_shielding=True,
        global_feat_dim=1,
    ).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return model

def eval_on_loader(test_loader_pth, ckpt_path):
    loader = torch.load(test_loader_pth, map_location=DEVICE)
    first_batch = next(iter(loader))
    num_node_features = first_batch.x.shape[1]
    model = load_gcn_from_ckpt(ckpt_path, num_node_features=num_node_features, device=DEVICE)

    trues, preds = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            pred = model(batch).detach().cpu().numpy()
            ytrue = batch.y.view(-1).detach().cpu().numpy()
            preds.append(pred)
            trues.append(ytrue)

    y_true = np.concatenate(trues)
    y_pred = np.concatenate(preds)
    return y_true, y_pred, regression_metrics(y_true, y_pred)

def parity(ax, y_true, y_pred, title, metrics, vmin, vmax):
    mae, rmse, r2 = metrics
    ax.scatter(y_true, y_pred, s=18, alpha=0.80, edgecolor='none')
    add_identity_line(ax, vmin, vmax)
    ax.set_xlim(vmin, vmax)
    ax.set_ylim(vmin, vmax)
    ax.set_title(title)
    ax.set_xlabel('True shift (ppm)')
    ax.set_ylabel('Predicted shift (ppm)')
    style_axes(ax, equal_aspect=True)
    annotate_panel_text(ax, metric_text(mae, rmse, r2, decimals=2))

y_true_A, y_pred_A, mA = eval_on_loader(GCN_TEST_LOADER_BEFORE, GCN_CKPT_BEFORE)
print_section('Figure input metrics')
print(metric_row('Before cleaning', *mA))
y_true_B, y_pred_B, mB = eval_on_loader(GCN_TEST_LOADER_AFTER, GCN_CKPT_AFTER)
print(metric_row('After cleaning', *mB))

vmin, vmax = compute_plot_limits(y_true_A, y_pred_A, y_true_B, y_pred_B)
fig, axes = create_figure('comparison', nrows=1, ncols=2, constrained_layout=True)
parity(axes[0], y_true_A, y_pred_A, 'GCN with shielding (before cleaning)', mA, vmin, vmax)
parity(axes[1], y_true_B, y_pred_B, 'GCN with shielding (after cleaning)', mB, vmin, vmax)

saved_paths = save_figure(fig, OUT_PNG)
plt.show()

report_saved_paths(saved_paths, 'Saved figure files')


### External Validation and Linear Calibration

Purpose: Evaluate the external 60 MHz dataset and visualize the effect of linear calibration.


In [ ]:
# -----------------------------------------------------------------------------
# Evaluate external transfer with linear calibration
# Purpose: Load the 60 MHz dataset, apply the pretrained model, fit the calibration line, and export the transfer plots.
# -----------------------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression

from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

SEED = 42
LAB_FEATURES = Path('data/processed/features_dataset_60M_NMR.parquet')
PRETRAINED_MODEL = Path('results/models/best_on_val_refined_ExtraTrees_with_shield.pkl')

OUT_PNG = Path('results/figures/Figure_LabTransfer_LinearCalibration_AB.png')
OUT_PNG.parent.mkdir(parents=True, exist_ok=True)

SMILES_COL = 'SMILES'
TARGET_COL = 'shift_value'

apply_plot_style()

def regression_metrics(y_true, y_pred):
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred))
    return mae, rmse, r2

def load_model(model_path: Path):
    loaded = joblib.load(model_path)
    if isinstance(loaded, dict):
        for k in ['model', 'best_model', 'best_estimator', 'best_estimator_', 'estimator']:
            if k in loaded:
                loaded = loaded[k]
                break
        else:
            raise TypeError(f'Loaded object is a dict but no known model key found. Available keys: {list(loaded.keys())}')
    return loaded

def parity(ax, y_true, y_pred, title, metrics, vmin, vmax, extra_text=None):
    mae, rmse, r2 = metrics
    ax.scatter(y_true, y_pred, s=18, alpha=0.80, edgecolor='none')
    add_identity_line(ax, vmin, vmax)
    ax.set_xlim(vmin, vmax)
    ax.set_ylim(vmin, vmax)
    ax.set_title(title)
    ax.set_xlabel('True lab shift (ppm)')
    ax.set_ylabel('Predicted shift (ppm)')
    style_axes(ax, equal_aspect=True)

    text = metric_text(mae, rmse, r2, decimals=2)
    if extra_text:
        text = extra_text + '\n' + text
    annotate_panel_text(ax, text)

df = pd.read_parquet(LAB_FEATURES)
if SMILES_COL not in df.columns:
    raise KeyError(f'Missing column: {SMILES_COL}')
if TARGET_COL not in df.columns:
    raise KeyError(f'Missing column: {TARGET_COL}')

smiles = df[SMILES_COL].astype(str).values
X = df.drop(columns=[SMILES_COL, TARGET_COL])
y_true = df[TARGET_COL].astype(float).values

model = load_model(PRETRAINED_MODEL)
y_pred_raw = model.predict(X)

lin = LinearRegression(fit_intercept=True)
lin.fit(y_pred_raw.reshape(-1, 1), y_true)
a = float(lin.coef_[0])
b = float(lin.intercept_)

y_pred_cal = a * y_pred_raw + b
m_raw = regression_metrics(y_true, y_pred_raw)
m_cal = regression_metrics(y_true, y_pred_cal)

err_raw = y_pred_raw - y_true
err_cal = y_pred_cal - y_true
err_table = pd.DataFrame({
    'SMILES': smiles,
    'True_shift_ppm': y_true,
    'Pred_raw_ppm': y_pred_raw,
    'Pred_cal_ppm': y_pred_cal,
    'Err_raw_ppm': err_raw,
    'AbsErr_raw_ppm': np.abs(err_raw),
    'Err_cal_ppm': err_cal,
    'AbsErr_cal_ppm': np.abs(err_cal),
}).sort_values('AbsErr_cal_ppm', ascending=False).reset_index(drop=True)

with pd.option_context('display.max_rows', 200, 'display.max_columns', 50, 'display.width', 200):
    print_section('Per-molecule error table (sorted by AbsErr_cal)')
    print(prepare_display_table(err_table))

vmin, vmax = compute_plot_limits(y_true, y_pred_raw, y_pred_cal)
fig, axes = create_figure('comparison', nrows=1, ncols=2, constrained_layout=True)
parity(axes[0], y_true, y_pred_raw, 'ExtraTrees -> Lab', m_raw, vmin, vmax)
parity(axes[1], y_true, y_pred_cal, 'ExtraTrees -> Lab + linear calibration', m_cal, vmin, vmax, extra_text=f'y = {a:.3f} * yhat + {b:.3f}')

saved_paths = save_figure(fig, OUT_PNG)
plt.show()

report_saved_paths(saved_paths, 'Saved figure files')
